# CV Project 2: Gradient Domain Editing & Geometric Transformations

**Objectives:**
- Part 1: Implement Poisson (Gradient Domain) image blending
- Part 2: Demonstrate geometric transformations (translation, rotation, scaling, affine, projective)
- Part 3: Paste an image onto a planar surface using projective transformation

In [ ]:
# Install dependencies (for Google Colab)
# !pip install numpy opencv-python scipy matplotlib scikit-image

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix, csr_matrix
from scipy.sparse.linalg import spsolve
import os

# For Google Colab: upload images
# from google.colab import files
# uploaded = files.upload()

os.makedirs('results', exist_ok=True)
print('Setup complete.')

---
## Part 1: Gradient Domain Editing (Poisson Image Blending)

### 1.1 Theory

**Problem:** Given a source image $g$ and target image $f^*$, blend an object from the source into the target seamlessly.

**Naive approach:** Directly copy pixels → visible seams at boundaries due to lighting/color differences.

**Poisson approach:** Instead of copying pixel values, we preserve the *gradients* of the source image while matching the *boundary values* of the target.

**Mathematical formulation:**

For a region $\Omega$ with boundary $\partial\Omega$, we minimize:

$$\min_f \iint_{\Omega} |\nabla f - \nabla g|^2 \quad \text{subject to} \quad f|_{\partial\Omega} = f^*|_{\partial\Omega}$$

This leads to the **Poisson equation**:

$$\nabla^2 f = \nabla^2 g \quad \text{inside } \Omega$$

**Discrete form:** For each pixel $p$ inside $\Omega$ with neighbors $N(p)$:

$$|N(p)| \cdot f_p - \sum_{q \in N(p) \cap \Omega} f_q = \sum_{q \in N(p)} (g_p - g_q) + \sum_{q \in N(p) \cap \partial\Omega} f^*_q$$

This creates a sparse linear system $Af = b$ solved efficiently with sparse solvers.

In [ ]:
# === Utility functions ===

def load_and_resize(path, target_shape=None):
    """Load image as RGB float [0,1]."""
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f'Cannot load: {path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if target_shape:
        img = cv2.resize(img, (target_shape[1], target_shape[0]))
    return img.astype(np.float64) / 255.0


def create_elliptical_mask(h, w, center=None, axes=None):
    """Create an elliptical binary mask."""
    if center is None:
        center = (w // 2, h // 2)
    if axes is None:
        axes = (w // 3, h // 3)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(mask, center, axes, 0, 0, 360, 255, -1)
    return mask

### 1.2 Implementation

In [ ]:
def get_neighbors(y, x):
    """4-connected neighbors."""
    return [(y-1, x), (y+1, x), (y, x-1), (y, x+1)]


def naive_copy_paste(source, target, mask, offset=(0, 0)):
    """Direct pixel copy (produces visible seams)."""
    result = target.copy()
    oy, ox = offset
    sh, sw = source.shape[:2]
    th, tw = target.shape[:2]
    mask_bool = mask > 0
    for y in range(sh):
        for x in range(sw):
            ty, tx = y + oy, x + ox
            if 0 <= ty < th and 0 <= tx < tw and mask_bool[y, x]:
                result[ty, tx] = source[y, x]
    return result


def poisson_blend_channel(source_ch, target_ch, mask, offset):
    """
    Solve Poisson equation for one channel.
    
    Builds sparse system Af = b where:
      A[i,i] = |N(p)|  (number of neighbors)
      A[i,j] = -1      (for neighbor j in Omega)
      b[i]   = sum of source gradients + boundary target values
    """
    oy, ox = offset
    sh, sw = source_ch.shape
    th, tw = target_ch.shape

    # Index all pixels inside Omega
    omega_pixels = []
    pixel_to_idx = {}
    for y in range(sh):
        for x in range(sw):
            if mask[y, x] > 0:
                ty, tx = y + oy, x + ox
                if 1 <= ty < th - 1 and 1 <= tx < tw - 1:
                    pixel_to_idx[(y, x)] = len(omega_pixels)
                    omega_pixels.append((y, x))

    n = len(omega_pixels)
    if n == 0:
        return target_ch.copy()

    A = lil_matrix((n, n), dtype=np.float64)
    b = np.zeros(n, dtype=np.float64)

    for idx, (y, x) in enumerate(omega_pixels):
        ty, tx = y + oy, x + ox
        num_nb = 0
        for ny, nx in get_neighbors(y, x):
            nty, ntx = ny + oy, nx + ox
            if 0 <= nty < th and 0 <= ntx < tw:
                num_nb += 1
                # Source gradient guidance
                if 0 <= ny < sh and 0 <= nx < sw:
                    b[idx] += source_ch[y, x] - source_ch[ny, nx]
                else:
                    b[idx] += source_ch[y, x]

                if (ny, nx) in pixel_to_idx:
                    A[idx, pixel_to_idx[(ny, nx)]] = -1
                else:
                    b[idx] += target_ch[nty, ntx]
        A[idx, idx] = num_nb

    # Solve sparse linear system
    A = csr_matrix(A)
    x_sol = spsolve(A, b)

    result = target_ch.copy()
    for idx, (y, xx) in enumerate(omega_pixels):
        result[y + oy, xx + ox] = np.clip(x_sol[idx], 0, 1)
    return result


def poisson_blend(source, target, mask, offset=(0, 0)):
    """Poisson blending for all 3 channels."""
    result = np.zeros_like(target)
    for c in range(3):
        print(f'  Channel {c}...')
        result[:, :, c] = poisson_blend_channel(
            source[:, :, c], target[:, :, c], mask, offset)
    return result

In [ ]:
def mixed_gradient_blend_channel(source_ch, target_ch, mask, offset):
    """
    Mixed gradient: use whichever gradient (source or target) has
    larger magnitude. Preserves strong edges from both images.
    """
    oy, ox = offset
    sh, sw = source_ch.shape
    th, tw = target_ch.shape

    omega_pixels = []
    pixel_to_idx = {}
    for y in range(sh):
        for x in range(sw):
            if mask[y, x] > 0:
                ty, tx = y + oy, x + ox
                if 1 <= ty < th - 1 and 1 <= tx < tw - 1:
                    pixel_to_idx[(y, x)] = len(omega_pixels)
                    omega_pixels.append((y, x))

    n = len(omega_pixels)
    if n == 0:
        return target_ch.copy()

    A = lil_matrix((n, n), dtype=np.float64)
    b = np.zeros(n, dtype=np.float64)

    for idx, (y, x) in enumerate(omega_pixels):
        ty, tx = y + oy, x + ox
        num_nb = 0
        for ny, nx in get_neighbors(y, x):
            nty, ntx = ny + oy, nx + ox
            if 0 <= nty < th and 0 <= ntx < tw:
                num_nb += 1
                # Pick gradient with larger magnitude
                gs = source_ch[y, x] - (source_ch[ny, nx] if 0 <= ny < sh and 0 <= nx < sw else 0)
                gt = target_ch[ty, tx] - target_ch[nty, ntx]
                b[idx] += gs if abs(gs) >= abs(gt) else gt

                if (ny, nx) in pixel_to_idx:
                    A[idx, pixel_to_idx[(ny, nx)]] = -1
                else:
                    b[idx] += target_ch[nty, ntx]
        A[idx, idx] = num_nb

    A = csr_matrix(A)
    x_sol = spsolve(A, b)

    result = target_ch.copy()
    for idx, (y, xx) in enumerate(omega_pixels):
        result[y + oy, xx + ox] = np.clip(x_sol[idx], 0, 1)
    return result


def mixed_gradient_blend(source, target, mask, offset=(0, 0)):
    """Mixed gradient blending for all 3 channels."""
    result = np.zeros_like(target)
    for c in range(3):
        print(f'  Channel {c}...')
        result[:, :, c] = mixed_gradient_blend_channel(
            source[:, :, c], target[:, :, c], mask, offset)
    return result

### 1.3 Generate Sample Images

If you have your own images, replace the paths below. Otherwise we generate synthetic test images.

In [ ]:
def generate_sample_images():
    """Generate synthetic source/target for demo."""
    # Target: landscape gradient
    target = np.zeros((400, 600, 3), dtype=np.float64)
    for y in range(400):
        r = y / 400.0
        if y < 200:
            target[y, :] = [0.3 + 0.4*r, 0.5 + 0.3*r, 0.9 - 0.2*r]
        else:
            target[y, :] = [0.2 + 0.1*r, 0.6 - 0.2*r, 0.1 + 0.1*r]

    # Source: bright object with dark surround
    source = np.zeros((150, 150, 3), dtype=np.float64)
    cy, cx = 75, 75
    for y in range(150):
        for x in range(150):
            d = np.sqrt((y-cy)**2 + (x-cx)**2)
            if d < 50:
                t = 1.0 - (d/50)*0.3
                source[y, x] = [t, t*0.85, t*0.2]
            else:
                source[y, x] = [0.1, 0.1, 0.15]

    mask = create_elliptical_mask(150, 150, (75, 75), (45, 45))
    return source, target, mask


# === Load or generate images ===
# To use your own images:
#   source = load_and_resize('images/source.jpg')
#   target = load_and_resize('images/background.jpg')
#   mask = cv2.imread('images/mask.png', 0)  # or create_elliptical_mask(...)

source, target, mask = generate_sample_images()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(target); axes[0].set_title('Target'); axes[0].axis('off')
axes[1].imshow(source); axes[1].set_title('Source'); axes[1].axis('off')
axes[2].imshow(mask, cmap='gray'); axes[2].set_title('Mask'); axes[2].axis('off')
plt.tight_layout()
plt.show()

### 1.4 Run Blending & Compare Results

In [ ]:
offset = (50, 200)

# 1. Naive copy-paste
print('Naive copy-paste...')
naive = naive_copy_paste(source, target, mask, offset)

# 2. Poisson blending
print('Poisson blending...')
poisson = poisson_blend(source, target, mask, offset)

# 3. Mixed gradient
print('Mixed gradient blending...')
mixed = mixed_gradient_blend(source, target, mask, offset)

# 4. OpenCV reference
print('OpenCV seamlessClone...')
src_u8 = (source * 255).astype(np.uint8)
tgt_u8 = (target * 255).astype(np.uint8)
oy, ox = offset
center = (ox + source.shape[1]//2, oy + source.shape[0]//2)
opencv_result = cv2.seamlessClone(
    cv2.cvtColor(src_u8, cv2.COLOR_RGB2BGR),
    cv2.cvtColor(tgt_u8, cv2.COLOR_RGB2BGR),
    mask, center, cv2.NORMAL_CLONE
)
opencv_result = cv2.cvtColor(opencv_result, cv2.COLOR_BGR2RGB) / 255.0

print('Done!')

In [ ]:
# === Visualization ===
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

axes[0, 0].imshow(np.clip(naive, 0, 1))
axes[0, 0].set_title('Naive Copy-Paste\n(visible seams)', fontsize=13)
axes[0, 0].axis('off')

axes[0, 1].imshow(np.clip(poisson, 0, 1))
axes[0, 1].set_title('Poisson Blending\n(seamless)', fontsize=13)
axes[0, 1].axis('off')

axes[1, 0].imshow(np.clip(mixed, 0, 1))
axes[1, 0].set_title('Mixed Gradient Blending\n(preserves strong edges)', fontsize=13)
axes[1, 0].axis('off')

axes[1, 1].imshow(np.clip(opencv_result, 0, 1))
axes[1, 1].set_title('OpenCV seamlessClone\n(reference)', fontsize=13)
axes[1, 1].axis('off')

plt.suptitle('Gradient Domain Editing: Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('results/part1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 2: Geometric Transformations

### 2.1 Theory

A 2D point $(x, y)$ in **homogeneous coordinates** is $(x, y, 1)$.
Transformations are $3 \times 3$ matrices:

$$\begin{pmatrix} x' \\ y' \\ w' \end{pmatrix} = \begin{pmatrix} m_{11} & m_{12} & m_{13} \\ m_{21} & m_{22} & m_{23} \\ m_{31} & m_{32} & m_{33} \end{pmatrix} \begin{pmatrix} x \\ y \\ 1 \end{pmatrix}$$

Final coordinates: $(x'/w', y'/w')$

| Transform | DOF | Matrix form | Preserves |
|-----------|-----|-------------|----------|
| Translation | 2 | $\begin{pmatrix}1&0&t_x\\0&1&t_y\\0&0&1\end{pmatrix}$ | Everything except position |
| Rotation | 1 | $\begin{pmatrix}\cos\theta&-\sin\theta&0\\\sin\theta&\cos\theta&0\\0&0&1\end{pmatrix}$ | Distances, angles |
| Scaling | 2 | $\begin{pmatrix}s_x&0&0\\0&s_y&0\\0&0&1\end{pmatrix}$ | Angles (if uniform) |
| **Affine** | **6** | $\begin{pmatrix}a&b&c\\d&e&f\\0&0&1\end{pmatrix}$ | **Parallel lines, ratios** |
| **Projective** | **8** | $\begin{pmatrix}h_1&h_2&h_3\\h_4&h_5&h_6\\h_7&h_8&1\end{pmatrix}$ | **Straight lines only** |

**Key difference:** Affine preserves parallelism. Projective does not (allows vanishing points).

In [ ]:
# === Transformation matrix builders ===

def translation_matrix(tx, ty):
    return np.array([[1,0,tx],[0,1,ty],[0,0,1]], dtype=np.float64)

def rotation_matrix(angle_deg, center=(0,0)):
    theta = np.radians(angle_deg)
    c, s = np.cos(theta), np.sin(theta)
    cx, cy = center
    T1 = translation_matrix(-cx, -cy)
    R = np.array([[c,-s,0],[s,c,0],[0,0,1]], dtype=np.float64)
    T2 = translation_matrix(cx, cy)
    return T2 @ R @ T1

def scaling_matrix(sx, sy, center=(0,0)):
    cx, cy = center
    T1 = translation_matrix(-cx, -cy)
    S = np.array([[sx,0,0],[0,sy,0],[0,0,1]], dtype=np.float64)
    T2 = translation_matrix(cx, cy)
    return T2 @ S @ T1

### 2.2 Generate Test Image

In [ ]:
def generate_checkerboard(size=400, squares=8):
    """Checkerboard pattern with markers for orientation."""
    img = np.zeros((size, size, 3), dtype=np.uint8)
    sq = size // squares
    colors = [(200,50,50), (50,200,50), (50,50,200), (200,200,50)]
    for i in range(squares):
        for j in range(squares):
            ci = (i+j) % 2
            color = colors[ci*2 + (i//(squares//2))]
            cv2.rectangle(img, (j*sq, i*sq), ((j+1)*sq, (i+1)*sq), color, -1)
    cv2.putText(img, 'TL', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    cv2.putText(img, 'TR', (size-50,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    cv2.putText(img, 'BL', (10,size-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    cv2.putText(img, 'BR', (size-50,size-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    cv2.circle(img, (size//2, size//2), size//6, (255,255,255), 3)
    return img

# Load user image or generate test pattern
if os.path.exists('images/input_transform.jpg'):
    img = cv2.cvtColor(cv2.imread('images/input_transform.jpg'), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (400, 400))
else:
    img = generate_checkerboard(400)

plt.figure(figsize=(5, 5))
plt.imshow(img); plt.title('Input Image'); plt.axis('off')
plt.show()

### 2.3 Apply All Transformations

In [ ]:
h, w = img.shape[:2]
center = (w/2, h/2)
out_size = (w + 100, h + 100)

transforms = {
    'Translation\n(tx=80, ty=50)': translation_matrix(80, 50),
    'Rotation\n(30\u00b0 center)': rotation_matrix(30, center),
    'Scaling\n(0.7x center)': scaling_matrix(0.7, 0.7, center),
    'Rot + Scale\n(20\u00b0, 0.8x)': scaling_matrix(0.8, 0.8, center) @ rotation_matrix(20, center),
}

# Affine: 3-point correspondence
src_pts = np.array([[50,50],[w-50,50],[50,h-50]], dtype=np.float32)
dst_pts = np.array([[70,80],[w-30,60],[100,h-30]], dtype=np.float32)
M_aff = cv2.getAffineTransform(src_pts, dst_pts)
transforms['Affine\n(shear+skew)'] = np.vstack([M_aff, [0,0,1]])

# Projective: 4-point correspondence
src_h = np.array([[0,0],[w,0],[w,h],[0,h]], dtype=np.float32)
dst_h = np.array([[50,30],[w-20,60],[w-80,h-20],[30,h-50]], dtype=np.float32)
H_proj, _ = cv2.findHomography(src_h, dst_h)
transforms['Projective\n(homography)'] = H_proj

# Display
fig, axes = plt.subplots(2, 4, figsize=(22, 12))
axes[0,0].imshow(img); axes[0,0].set_title('Original', fontsize=13, fontweight='bold'); axes[0,0].axis('off')

for idx, (name, H) in enumerate(transforms.items()):
    r, c = (idx+1)//4, (idx+1)%4
    result = cv2.warpPerspective(img, H, out_size, borderValue=(30,30,30))
    axes[r,c].imshow(result); axes[r,c].set_title(name, fontsize=11); axes[r,c].axis('off')

axes[1,3].axis('off')
plt.suptitle('Geometric Transformations', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('results/part2_all_transforms.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.4 Affine vs Projective: Detailed Comparison

In [ ]:
# Same target corners, but affine uses only 3 points
src_corners = np.array([[0,0],[w,0],[w,h],[0,h]], dtype=np.float32)
dst_corners = np.array([[80,60],[w-80,60],[w+20,h+20],[-20,h+20]], dtype=np.float32)

# Affine from first 3 points
M_aff2 = cv2.getAffineTransform(src_corners[:3], dst_corners[:3])
H_aff2 = np.vstack([M_aff2, [0,0,1]])

# Projective from all 4 points
H_proj2, _ = cv2.findHomography(src_corners, dst_corners)

res_aff = cv2.warpPerspective(img, H_aff2, (w+100,h+100), borderValue=(30,30,30))
res_proj = cv2.warpPerspective(img, H_proj2, (w+100,h+100), borderValue=(30,30,30))

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
axes[0].imshow(img); axes[0].set_title('Original', fontsize=13); axes[0].axis('off')
axes[1].imshow(res_aff); axes[1].set_title('Affine\n(parallel lines PRESERVED)', fontsize=12); axes[1].axis('off')
axes[2].imshow(res_proj); axes[2].set_title('Projective\n(parallel lines NOT preserved)', fontsize=12); axes[2].axis('off')
plt.suptitle('Affine vs Projective', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('results/part2_affine_vs_projective.png', dpi=150, bbox_inches='tight')
plt.show()

print('Affine matrix (last row = [0, 0, 1]):')
print(np.round(H_aff2, 4))
print('\nProjective matrix (last row != [0, 0, 1]):')
print(np.round(H_proj2, 4))

In [ ]:
# Grid deformation visualization
def visualize_grid(H, title, ax, grid_size=10, extent=200):
    for i in range(0, extent+1, grid_size):
        ax.plot([0,extent],[i,i], 'lightgray', lw=0.5)
        ax.plot([i,i],[0,extent], 'lightgray', lw=0.5)
    for i in range(0, extent+1, grid_size):
        pts = np.array([[x,i,1] for x in range(0,extent+1,2)])
        w2 = (H @ pts.T).T; w2 = w2[:,:2]/w2[:,2:3]
        ax.plot(w2[:,0], w2[:,1], 'b-', lw=0.8)
        pts = np.array([[i,y,1] for y in range(0,extent+1,2)])
        w2 = (H @ pts.T).T; w2 = w2[:,:2]/w2[:,2:3]
        ax.plot(w2[:,0], w2[:,1], 'r-', lw=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_aspect('equal')
    ax.set_xlim(-50, extent+100); ax.set_ylim(-50, extent+100)
    ax.invert_yaxis()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
visualize_grid(np.eye(3), 'Original Grid', axes[0])
visualize_grid(H_aff2, 'Affine\n(parallel lines kept)', axes[1])
visualize_grid(H_proj2, 'Projective\n(perspective distortion)', axes[2])
plt.suptitle('Grid Deformation Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/part2_grid_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 3: Projective Billboard Pasting (Extended)

### 3.1 Theory

Given a content image and 4 target corner points on a planar surface in a scene:

1. Define source corners as the content image corners: $(0,0), (w,0), (w,h), (0,h)$
2. Compute homography $H$ mapping source to target corners using DLT
3. Warp content with $H$ and composite into the scene

**DLT Algorithm:** For each correspondence $(x,y) \to (x',y')$, we get 2 equations:

$$\begin{pmatrix}-x & -y & -1 & 0 & 0 & 0 & x'x & x'y & x' \\ 0 & 0 & 0 & -x & -y & -1 & y'x & y'y & y'\end{pmatrix} \mathbf{h} = \mathbf{0}$$

Stack all equations into $A\mathbf{h} = 0$ and solve via SVD (last row of $V^T$).

In [ ]:
def compute_homography_dlt(src_pts, dst_pts):
    """Manual DLT implementation for educational purposes."""
    n = len(src_pts)
    A = np.zeros((2*n, 9))
    for i in range(n):
        x, y = src_pts[i]
        xp, yp = dst_pts[i]
        A[2*i]   = [-x, -y, -1,  0,  0,  0, xp*x, xp*y, xp]
        A[2*i+1] = [ 0,  0,  0, -x, -y, -1, yp*x, yp*y, yp]
    _, _, Vt = np.linalg.svd(A)
    H = Vt[-1].reshape(3, 3)
    return H / H[2, 2]


def paste_on_surface(scene, content, target_corners, blend_mode='replace'):
    """Warp content onto a planar surface in the scene."""
    ch, cw = content.shape[:2]
    sh, sw = scene.shape[:2]
    src_corners = np.array([[0,0],[cw-1,0],[cw-1,ch-1],[0,ch-1]], dtype=np.float32)
    target_corners = np.array(target_corners, dtype=np.float32)

    H = cv2.getPerspectiveTransform(src_corners, target_corners)
    warped = cv2.warpPerspective(content, H, (sw, sh))
    mask = cv2.warpPerspective(np.ones((ch,cw), dtype=np.uint8)*255, H, (sw, sh))

    result = scene.copy()
    if blend_mode == 'alpha':
        kern = np.ones((5,5), np.uint8)
        eroded = cv2.erode(mask, kern, iterations=3)
        alpha = cv2.GaussianBlur(eroded, (21,21), 10).astype(np.float32) / 255.0
        for c in range(3):
            result[:,:,c] = (alpha*warped[:,:,c] + (1-alpha)*scene[:,:,c]).astype(np.uint8)
    else:
        m = mask > 0
        for c in range(3):
            result[:,:,c][m] = warped[:,:,c][m]
    return result, warped, mask

In [ ]:
# === Generate scene and content images ===

def generate_scene_image(size=(600, 800)):
    h, w = size
    scene = np.zeros((h, w, 3), dtype=np.uint8)
    for y in range(h//2):
        r = y/(h//2)
        scene[y,:] = [int(180-80*r), int(200-60*r), int(240-40*r)]
    for y in range(h//2, h):
        r = (y-h//2)/(h//2)
        scene[y,:] = [int(80+30*r), int(120+20*r), int(80+30*r)]
    cv2.rectangle(scene, (100,100), (700,500), (160,150,140), -1)
    cv2.rectangle(scene, (100,100), (700,500), (100,90,80), 3)
    for wx in [150,300,500,600]:
        for wy in [150,280,380]:
            cv2.rectangle(scene, (wx,wy), (wx+60,wy+50), (200,220,240), -1)
    cv2.rectangle(scene, (195,125), (555,345), (60,60,60), 5)
    return scene

def generate_content_image(size=(300, 500)):
    h, w = size
    content = np.zeros((h, w, 3), dtype=np.uint8)
    for y in range(h):
        r = y/h
        content[y,:] = [int(30+200*r), int(50+100*(1-r)), int(200-150*r)]
    cv2.putText(content, 'HELLO', (w//4-30,h//3), cv2.FONT_HERSHEY_SIMPLEX, 2.0, (255,255,255), 4)
    cv2.putText(content, 'WORLD!', (w//4-20,2*h//3), cv2.FONT_HERSHEY_SIMPLEX, 1.8, (255,255,0), 3)
    return content

if os.path.exists('images/scene.jpg') and os.path.exists('images/content.jpg'):
    scene = cv2.cvtColor(cv2.imread('images/scene.jpg'), cv2.COLOR_BGR2RGB)
    content = cv2.cvtColor(cv2.imread('images/content.jpg'), cv2.COLOR_BGR2RGB)
else:
    scene = cv2.cvtColor(generate_scene_image(), cv2.COLOR_BGR2RGB)
    content = cv2.cvtColor(generate_content_image(), cv2.COLOR_BGR2RGB)

# Billboard corners
frontal_corners = np.array([[200,130],[550,130],[550,340],[200,340]], dtype=np.float32)
perspective_corners = frontal_corners.copy()
perspective_corners[0,0] += 40; perspective_corners[1,0] -= 40
perspective_corners[0,1] += 15; perspective_corners[1,1] += 15

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(scene); ax[0].set_title('Scene'); ax[0].axis('off')
ax[1].imshow(content); ax[1].set_title('Content'); ax[1].axis('off')
plt.show()

In [ ]:
# === Run billboard pasting ===
print('Frontal paste...')
res_frontal, _, _ = paste_on_surface(scene, content, frontal_corners)

print('Perspective paste...')
res_persp, _, _ = paste_on_surface(scene, content, perspective_corners)

print('Perspective + alpha blend...')
res_blend, _, _ = paste_on_surface(scene, content, perspective_corners, 'alpha')

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

scene_marked = scene.copy()
pts = frontal_corners.astype(np.int32)
for i in range(4):
    cv2.circle(scene_marked, tuple(pts[i]), 6, (255,0,0), -1)
    cv2.line(scene_marked, tuple(pts[i]), tuple(pts[(i+1)%4]), (255,0,0), 2)
axes[0,0].imshow(scene_marked)
axes[0,0].set_title('Target Region', fontsize=13); axes[0,0].axis('off')

axes[0,1].imshow(res_frontal)
axes[0,1].set_title('Frontal Paste', fontsize=13); axes[0,1].axis('off')

axes[1,0].imshow(res_persp)
axes[1,0].set_title('Perspective Paste', fontsize=13); axes[1,0].axis('off')

axes[1,1].imshow(res_blend)
axes[1,1].set_title('Perspective + Alpha Blend', fontsize=13); axes[1,1].axis('off')

plt.suptitle('Projective Billboard Pasting', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('results/part3_billboard.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Verify DLT vs OpenCV homography ===
ch, cw = content.shape[:2]
src_c = np.array([[0,0],[cw-1,0],[cw-1,ch-1],[0,ch-1]], dtype=np.float32)

H_cv = cv2.getPerspectiveTransform(src_c, perspective_corners)
H_dlt = compute_homography_dlt(src_c, perspective_corners)

print('Homography (OpenCV):')
print(np.round(H_cv, 4))
print('\nHomography (Manual DLT):')
print(np.round(H_dlt, 4))
print('\nMax difference:', np.max(np.abs(H_cv - H_dlt)))

---
## Summary

### Part 1: Gradient Domain Editing
- **Naive copy-paste** produces visible seams at boundaries
- **Poisson blending** solves a sparse linear system to match gradients, producing seamless results
- **Mixed gradient** variant preserves strong edges from both source and target

### Part 2: Geometric Transformations
- **Translation, rotation, scaling** are basic rigid/similarity transforms
- **Affine** (6 DOF): preserves parallel lines, needs 3 point pairs
- **Projective** (8 DOF): preserves straight lines only, needs 4 point pairs
- Key insight: affine is a special case of projective where the last matrix row is $[0, 0, 1]$

### Part 3: Billboard Pasting
- Projective transformation maps rectangular content onto arbitrary quadrilaterals
- DLT algorithm computes homography from 4 point correspondences
- Alpha blending at edges improves visual quality